# geo-graphs: the Stage 1 pipeline, end to end

This walks through every capability that exists today, framed by the job each
one does in the project rather than by its API.

**This is not a test harness.** Every module below is a production component of
Stage 1 road extraction. What is missing is the model, and only the model:

```
TRAINING                                        INFERENCE

OSM graph                                       satellite image
    │ osm.ground_truth_graph                        │
    ▼                                               │ ┌───────────────┐
road graph  ──── raster.rasterize ────▶ mask ───────┼─│    U-Net      │  ◀── not built yet
    │                                (target)       │ └───────────────┘
    │                                               ▼
    │                                        predicted mask
    │                                               │ skeleton.graph_from_mask
    │                                               │ cleanup.clean
    │                                               ▼
    └────────────── metrics.apls ──────────── predicted graph
```

`raster.rasterize` produces the U-Net's *training targets*. `skeleton` +
`cleanup` are the *inference-time decoder* that turns predictions back into a
graph. Swap the perfect mask below for `unet(image)` and the pipeline is
complete.

Ground truth comes from OpenStreetMap directly, so all of this runs with no
imagery download. OSM responses are cached under `./cache`, so the first run of
a new tile is slow and the rest are instant.

In [ ]:
import networkx as nx
import numpy as np
import plotly.graph_objects as go

from geo_graphs import cleanup, geograph, metrics, raster, roundtrip, skeleton, tiles
from geo_graphs.osm import ground_truth_graph

# Downtown Las Vegas: inside SpaceNet AOI 2, so anything tuned here still
# means something once real imagery arrives.
VEGAS = (36.1699, -115.1398)

## 1. Tiles — the coordinate system everything shares

A `Tile` is a patch of ground plus the raster grid laid over it. Two decisions
in here matter for everything downstream.

**Resolution is 1 m/px.** That makes one pixel one metre, so graph lengths and
APLS distances are already in metres and never need converting. It also matches
SpaceNet, so the numbers stay comparable.

**Everything works in pixel coordinates.** Ground truth is projected to UTM and
then to pixels once, at ingestion. After that, the mask, the model output, the
recovered graph and the metric all live in the same frame — no CRS handling
leaks into the model or the scoring code.

In [ ]:
tile = tiles.tile_from_center(*VEGAS, size_m=512)

print(f"CRS         {tile.crs.name}")
print(f"grid        {tiles.shape(tile)} px at {tile.resolution} m/px")
print(f"bounds      {[round(v, 4) for v in tiles.lonlat_bounds(tile)]}")

# Pixel <-> world is an exact round trip; (col, row) with row increasing south.
corner = tiles.px_to_world(tile, np.array([0.0, 0.0]))
print(f"px (0,0) is the NW corner: {np.allclose(corner, [tile.x_min, tile.y_max])}")

## 2. Ground truth from OSM — the label source

`ground_truth_graph` fetches roads covering the tile, projects them, and clips
them to the tile boundary. This is where labels come from.

Two details that turned out to matter:

- OSM arrives **directed**, so a two-way street is two edges. They are collapsed
  to undirected, or every street would be rasterized and scored twice.
- The query is padded beyond the tile, then clipped locally. Roads crossing the
  edge arrive whole and get cut cleanly, rather than ending wherever OSM's own
  bbox filter happened to truncate them.

In [ ]:
truth = ground_truth_graph(tile)

print(f"nodes       {truth.number_of_nodes()}")
print(f"edges       {truth.number_of_edges()}")
print(f"road length {geograph.total_length(truth):,.0f} m")
print(f"components  {nx.number_connected_components(truth)}")

# Graph conventions: nodes carry `pos` (x, y) in pixels, edges carry the
# polyline `pts` and its `length`.
u, v, k = next(iter(truth.edges(keys=True)))
print(f"\nexample edge {u}->{v}: {len(truth.edges[u, v, k]['pts'])} vertices, "
      f"{truth.edges[u, v, k]['length']:.1f} m")

In [ ]:
def graph_trace(G, color, name, width=2):
    """One plotly trace for a whole graph, edges separated by None gaps."""
    xs, ys = [], []
    for a, b, key in G.edges(keys=True):
        pts = geograph.oriented_pts(G, a, b, key)
        xs.extend([*pts[:, 0].tolist(), None])
        ys.extend([*pts[:, 1].tolist(), None])
    return go.Scatter(x=xs, y=ys, mode="lines", name=name,
                      line=dict(color=color, width=width))


def square(fig, title):
    """Pixel-space axes: equal aspect, north up."""
    fig.update_layout(title=title, width=620, height=620,
                      margin=dict(l=40, r=40, t=60, b=40))
    fig.update_yaxes(autorange="reversed", scaleanchor="x", scaleratio=1)
    return fig


square(go.Figure([graph_trace(truth, "#2b6cb0", "OSM ground truth")]),
       "Ground-truth road graph (pixel coordinates)").show()

## 3. Rasterize — the U-Net's training target

`raster.rasterize` draws edge centerlines and dilates them to the width of a
road surface. **This is the label the segmentation model is trained against**,
built the same way SpaceNet builds its own: a centerline buffered by a fixed
radius.

`half_width_px` is the road width knob. At 1 m/px, the default 2 gives roads
about 5 m wide.

In [ ]:
mask = raster.rasterize(truth, tile)

print(f"mask shape  {mask.shape}, dtype {mask.dtype}")
print(f"road pixels {mask.mean():.1%} of the tile")

square(go.Figure(go.Heatmap(z=mask.astype(np.uint8), showscale=False,
                            colorscale=[[0, "#ffffff"], [1, "#2b6cb0"]])),
       "Training target: rasterized road mask").show()

## 4. Mask → graph — the inference-time decoder

This is the half that runs on *model predictions*. Nothing here knows or cares
whether the mask came from OSM or from a network.

1. `skeletonize` thins the mask to one-pixel-wide centerlines.
2. Classify by 8-connected neighbour count: 1 = endpoint, 2 = interior,
   ≥3 = junction. Junction blobs collapse to one node at their centroid.
3. Trace along degree-2 chains between nodes to recover edge polylines.
4. `cleanup.clean` simplifies (Douglas-Peucker), prunes short spurs, snaps
   nearby junctions, then prunes again — snapping can strand a new spur.

Here we feed it a *perfect* mask, so whatever it loses is the pipeline's own
error rather than the model's.

In [ ]:
raw = skeleton.graph_from_mask(mask)
recovered = cleanup.clean(raw)

for name, G in [("truth", truth), ("raw skeleton", raw), ("cleaned", recovered)]:
    print(f"{name:14s} {G.number_of_nodes():4d} nodes  {G.number_of_edges():4d} edges  "
          f"{geograph.total_length(G):8,.0f} m")

In [ ]:
square(go.Figure([
    graph_trace(truth, "#2b6cb0", "ground truth", width=4),
    graph_trace(recovered, "#e53e3e", "recovered from mask", width=1.5),
]), "Recovered graph over ground truth").show()

## 5. Scoring — IoU and APLS

Two metrics, measuring different things.

**IoU** compares pixels. It is what a segmentation loss optimizes and says
nothing about connectivity.

**APLS** compares *routes*. It samples control points along the network, injects
each into the other graph, and compares shortest-path distances between every
pair. The two directions catch different failures — `gt→prop` punishes roads you
missed, `prop→gt` punishes roads you invented — and they are combined with a
harmonic mean, so a proposal has to do both.

APLS here is validated against the SpaceNet reference implementation to within
3e-5; see `tests/test_against_reference.py`.

In [ ]:
score = metrics.apls(truth, recovered)

print(f"mask IoU              {metrics.iou(mask, raster.rasterize(recovered, tile)):.4f}")
print(f"APLS                  {score.score:.4f}")
print(f"  gt->prop (missed)   {score.gt_to_prop:.4f}")
print(f"  prop->gt (invented) {score.prop_to_gt:.4f}")
print(f"  control points      {score.n_control}")

## 6. Why a pixel metric is the wrong target

This is the point of the whole project, and it is worth seeing rather than being
told.

Punch small gaps into the *perfect* mask — the kind a tree shadow or an overpass
produces — and watch the two metrics diverge. Each gap costs a handful of pixels
and severs a route.

Averaged over several random placements, because where a gap lands matters: one
in the middle of a through-road is far more damaging than one on a dead end.

In [ ]:
def punch_gaps(mask, n_gaps, seed, radius=2):
    """Erase small squares at random road pixels."""
    rng = np.random.default_rng(seed)
    out = mask.copy()
    rows, cols = np.nonzero(mask)
    for i in rng.choice(len(rows), n_gaps, replace=False):
        y, x = rows[i], cols[i]
        out[max(0, y - radius):y + radius + 1, max(0, x - radius):x + radius + 1] = False
    return out


GAP_COUNTS = (0, 5, 10, 20, 40, 80)
SEEDS = range(5)

rows = []
for n_gaps in GAP_COUNTS:
    ious, aplss = [], []
    for seed in SEEDS:
        damaged = punch_gaps(mask, n_gaps, seed)
        proposal = cleanup.clean(skeleton.graph_from_mask(damaged))
        ious.append(metrics.iou(mask, damaged))
        aplss.append(metrics.apls(truth, proposal).score)
    rows.append((n_gaps, float(np.mean(ious)), float(np.mean(aplss))))

print(f"{'gaps':>5s} {'IoU':>8s} {'APLS':>8s}   (mean of {len(SEEDS)} seeds)")
for n_gaps, iou, apls in rows:
    print(f"{n_gaps:5d} {iou:8.4f} {apls:8.4f}")

In [ ]:
gaps, ious, aplss = map(list, zip(*rows, strict=True))

fig = go.Figure([
    go.Scatter(x=gaps, y=ious, name="pixel IoU", mode="lines+markers",
               line=dict(color="#38a169", width=3)),
    go.Scatter(x=gaps, y=aplss, name="APLS (topology)", mode="lines+markers",
               line=dict(color="#e53e3e", width=3)),
])
fig.update_layout(
    title="A mask that still looks right produces a graph that is badly wrong",
    xaxis_title="number of small gaps punched into a perfect mask",
    yaxis_title="score", yaxis_range=[0, 1.05],
    width=760, height=440, margin=dict(l=60, r=40, t=70, b=60),
)
fig.show()

print(f"At {gaps[-1]} gaps: {1 - ious[-1]:.1%} of pixels lost, "
      f"{1 - aplss[-1] / aplss[0]:.1%} of the topology score lost.")

That gap is the entire argument for the project. A segmentation model optimizing
pixel loss can look excellent and still produce an unusable graph, and no amount
of staring at IoU will reveal it.

It is also the reason the evaluation harness was built *before* the model: a
number you do not trust cannot tell you whether a model is working.

## 7. The ceiling — why a perfect mask does not score 1.0

Step 5 scored below 1.0 on a *perfect* mask. That is not a bug and not a tuning
failure.

A 2D binary mask **cannot represent a non-planar network**. Where a road bridges
over another, OSM correctly records no junction — you cannot turn from one to
the other. Flattened to a plane the two roads touch, and skeletonization has no
choice but to emit a junction that does not exist.

Rather than assert that, measure it. Widening the tile pulls in more of the
freeway interchange north of downtown, and with it more grade separation. If
non-planarity really drives the ceiling, the score should fall as the crossing
count rises — and the damage should land specifically in the `prop→gt`
direction, the one that punishes *invented* connections.

`roundtrip.run` is the pipeline's own entry point for this measurement. Every
Stage 1 model number inherits whatever it reports.

In [ ]:
from shapely.geometry import LineString


def non_planar_crossings(G):
    """Edge pairs that cross geometrically while sharing no node.

    Each one is a place where two roads pass at different heights, and where a
    flattened mask has no way to say so.
    """
    lines = {key: LineString(geograph.oriented_pts(G, *key)) for key in G.edges(keys=True)}
    return sum(
        1
        for a, la in lines.items()
        for b, lb in lines.items()
        if a < b and not set(a[:2]) & set(b[:2]) and la.crosses(lb)
    )


header = f"{'tile':>7s} {'km road':>8s} {'crossings':>10s} {'per km':>7s} {'APLS':>8s} {'prop->gt':>9s}"
print(header)
print("-" * len(header))
for size_m in (512.0, 1024.0, 2048.0):
    truth_at = ground_truth_graph(tiles.tile_from_center(*VEGAS, size_m=size_m))
    report = roundtrip.run(*VEGAS, size_m=size_m)
    km = geograph.total_length(truth_at) / 1000.0
    crossings = non_planar_crossings(truth_at)
    print(f"{size_m:6.0f}m {km:8.1f} {crossings:10d} {crossings / km:7.2f} "
          f"{report.apls:8.4f} {report.apls_prop_to_gt:9.4f}")

Read that table carefully, because the obvious column is the wrong one.

**Crossing count does not predict the ceiling.** The 2048 m tile has twice as
many crossings as the 1024 m tile and still scores higher. Counting is the
coarse-grained metric here, and on its own it would have sent us looking for a
different explanation entirely.

**Crossing density does.** Normalize by road length and the ordering is exact:
0.00, 0.56 and 1.03 crossings per km map to 0.9624, 0.9487 and 0.8764. That
makes mechanical sense — APLS averages over route pairs, so the same twenty
crossings diluted across 71 km of road damage a far smaller share of routes than
concentrated in 19 km. And the loss lands in `prop→gt` every time, the direction
that punishes *invented* connections, which is the specific signature an
overpass should leave.

Being honest about how strong this is: three tiles is three points, and tile size
changes more than crossing density alone. The direction, the magnitude and the
location of the damage all agree, which is enough to act on but not a proof. The
independent evidence is the sweep in `EXPERIMENT_LOG.md`: moving road width from
1.5 m to 18 m and resolution from 0.5 to 2 m/px shifts the score only between
0.861 and 0.901, so this is not something thinner roads or a finer grid will fix.

Note also that the ceiling is below 1.0 even at zero crossings. Rasterizing and
thinning costs a little geometry regardless — junction blobs collapse to a
centroid, coordinates quantize.

This is the measured argument for Stage 2. Sat2Graph-style per-cell directional
edge slots can encode two roads meeting at one pixel *without* a junction, which
a mask simply cannot express.

## 8. Where the model plugs in

Everything above runs on a mask. The only change training introduces is where
that mask comes from:

```python
# today
mask = raster.rasterize(truth, tile)          # perfect labels

# with a model
mask = unet(image).sigmoid() > threshold      # predictions
```

and the rest is unchanged:

```python
proposal = cleanup.clean(skeleton.graph_from_mask(mask))
score = metrics.apls(truth, proposal)
```

So the remaining Stage 1 work is:

1. **Imagery.** SpaceNet Roads from S3, tiled with `tiles`. `raster.rasterize`
   already produces the matching targets.
2. **The U-Net.** Binary cross-entropy plus soft Dice on 256×256 crops.
3. **Threshold selection.** The `> threshold` above is a real hyperparameter,
   and section 6 says to tune it on APLS rather than IoU.

Worth carrying into that work: the decoder and the metric are already validated,
so when a trained model scores badly you will know the fault is the model. That
is the whole reason this half was built first.

See `EXPERIMENT_LOG.md` for the measurements behind these claims and
`BACKLOG.md` for what is deliberately deferred.